# 06 — Dados de 2026 via API do FIRMS (tarefa A9)
Os arquivos anuais por país do FIRMS (usados no notebook 01) só vão até **2025**. Para 2026 usamos a **API do FIRMS**, que exige uma chave gratuita (**MAP_KEY**).

**Saída:** `data/processed/harmonized_2026.parquet`, no mesmo contrato do `harmonized.parquet`:
`lat, lon, ano, mes, focos_harmonizados, fonte='viirs', dado_incompleto` (só linhas com focos > 0).

**Processamento igual ao do projeto (notebooks 01 e 04):** VIIRS S-NPP · `confidence ∈ {n, h}` · remoção das células de 0,02° da máscara de fontes estáticas (`static_mask.parquet`) · célula 0,25° × mês.

**Como rodar:**
1. Peça a chave em https://firms.modaps.eosdis.nasa.gov/api/map_key/ (chega por e-mail).
2. Crie um arquivo `.env` na **raiz do repositório** com a linha `FIRMS_MAP_KEY=sua_chave` (o `.env` já está no `.gitignore`), ou defina a variável de ambiente `FIRMS_MAP_KEY`.
3. Rode o notebook. Sem chave, ele só imprime as instruções e para sem erro.
4. Para juntar 2026 ao `harmonized.parquet`, mude `ANEXAR = True` na célula abaixo e rode de novo.

In [1]:
ANEXAR = True    # True = junta harmonized_2026.parquet ao harmonized.parquet (última célula)

import io, os, time, json, calendar
from datetime import date, timedelta
from pathlib import Path
import duckdb, pandas as pd, requests

RAW, PROC = Path('../data/raw'), Path('../data/processed')
ANO = 2026
INICIO = date(ANO, 1, 1)
HOJE = date.today()
FIM = min(HOJE, date(ANO, 12, 31))
API = 'https://firms.modaps.eosdis.nasa.gov/api'
BBOX = '-75,-35,-28,6'           # oeste,sul,leste,norte: Brasil + ilhas oceânicas (o endpoint por país está fora do ar)
DAY_RANGE_MAX = 5                # limite da API (1 a 5 dias por chamada)
CELULA, FINO = 0.25, 0.02        # grade do calendário / célula da máscara (iguais ao notebook 04)
CELULA_BR = 0.1                  # pegada do Brasil (ver seção 3)
COLS = ['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite', 'confidence', 'frp', 'daynight', 'type']


def ler_chave():
    """Lê FIRMS_MAP_KEY do ambiente ou do .env na raiz. Nunca imprime a chave."""
    k = os.environ.get('FIRMS_MAP_KEY', '').strip()
    if k:
        return k
    for env in (Path('../.env'), Path('.env')):
        if env.exists():
            for linha in env.read_text(encoding='utf-8').splitlines():
                linha = linha.strip()
                if linha.startswith('FIRMS_MAP_KEY') and '=' in linha:
                    return linha.split('=', 1)[1].strip().strip('"').strip("'")
    return ''


KEY = ler_chave()
if KEY:
    print(f'MAP_KEY encontrada ({len(KEY)} caracteres). Período: {INICIO} → {FIM}.')
else:
    print("""MAP_KEY do FIRMS NÃO encontrada — o notebook vai parar aqui, sem erro.

Para baixar os focos de 2026:
  1. Peça uma chave gratuita em https://firms.modaps.eosdis.nasa.gov/api/map_key/ (chega por e-mail).
  2. Na raiz do repositório, crie o arquivo .env com a linha:
         FIRMS_MAP_KEY=sua_chave_aqui
     (o .env já está no .gitignore; nunca faça commit da chave)
     Alternativa: defina a variável de ambiente FIRMS_MAP_KEY antes de abrir o Jupyter.
  3. Rode este notebook de novo.""")

MAP_KEY encontrada (32 caracteres). Período: 2026-01-01 → 2026-11-14.


## 1 — O que a API oferece (documentação oficial, conferida em 14/11/2026)
- **Área:** `/api/area/csv/[MAP_KEY]/[SOURCE]/[AREA]/[DAY_RANGE]/[DATE]` — `AREA` = `oeste,sul,leste,norte`; `DAY_RANGE` de **1 a 5** dias; `DATE` = `AAAA-MM-DD` (dados a partir dessa data). https://firms.modaps.eosdis.nasa.gov/api/area/
- **País:** `/api/country/...` aparece em https://firms.modaps.eosdis.nasa.gov/api/ como *"Feature currently not available"*. Por isso usamos **área** (retângulo do Brasil) e depois recortamos o Brasil (seção 3).
- **Fontes VIIRS S-NPP:** `VIIRS_SNPP_SP` (*standard processing*, versão definitiva, mesma dos arquivos anuais) e `VIIRS_SNPP_NRT` (*near real-time*, provisória; não tem a coluna `type`).
- **Disponibilidade:** `/api/data_availability/csv/[MAP_KEY]/[SENSOR]` diz até que data existe SP. Usamos **SP até essa data e NRT só depois dela** (sem sobreposição).
- **Limite:** 5000 transações a cada 10 minutos por MAP_KEY; pedidos grandes contam como várias. Aqui são ~64 chamadas (318 dias ÷ 5).

In [2]:
def _limpa(txt):
    """Remove a chave de qualquer mensagem antes de imprimir."""
    return str(txt).replace(KEY, '***') if KEY else str(txt)


def get_csv(caminho, tentativas=5):
    """GET em API/caminho; devolve DataFrame (pode ser vazio). Retentativas com espera crescente."""
    url = f'{API}/{caminho}'
    erro = ''
    for t in range(tentativas):
        try:
            r = requests.get(url, timeout=120)
            txt = r.text
            primeira = txt.lstrip().split(chr(10), 1)[0].lower()
            if r.status_code == 200 and ',' in primeira and 'invalid' not in primeira:
                return pd.read_csv(io.StringIO(txt), dtype=str)
            if r.status_code == 200 and not txt.strip():
                return pd.DataFrame()
            erro = f'HTTP {r.status_code}: {txt.strip()[:200]}'
            if 'invalid' in txt.lower() and 'key' in txt.lower():
                raise RuntimeError('MAP_KEY inválida. Confira o valor no .env.')
        except requests.RequestException as e:
            erro = type(e).__name__
        espera = 60 if ('exceed' in erro.lower() or '429' in erro) else 2 ** (t + 1)
        print(f'  tentativa {t + 1} falhou ({_limpa(erro)}); esperando {espera}s')
        time.sleep(espera)
    raise RuntimeError(_limpa(f'Falhou após {tentativas} tentativas: {erro}'))


SP_FIM = None
if KEY:
    disp = get_csv(f'data_availability/csv/{KEY}/VIIRS_SNPP_SP')
    display(disp)
    col_max = next((c for c in disp.columns if 'max' in c.lower()), None)
    if col_max is not None and len(disp):
        linha = disp[disp.iloc[:, 0].str.upper() == 'VIIRS_SNPP_SP']
        linha = linha if len(linha) else disp
        SP_FIM = pd.to_datetime(linha[col_max].iloc[0]).date()
    print(f'VIIRS_SNPP_SP disponível até: {SP_FIM}  →  NRT a partir de: {(SP_FIM or INICIO - timedelta(1)) + timedelta(1)}')
else:
    print('Sem MAP_KEY: pulando.')

,data_id,min_date,max_date
0,VIIRS_SNPP_SP,2012-01-20,2026-07-31


VIIRS_SNPP_SP disponível até: 2026-07-31  →  NRT a partir de: 2026-08-01


## 2 — Download (blocos de até 5 dias, com retentativas)
Os dados brutos vão para `data/raw/ano_corrente/viirs-snpp_2026.parquet` (pasta separada para não entrar no glob do notebook 04) (fora do git), **com as mesmas colunas e tipos dos anos anteriores** (o `type` fica vazio nos dias NRT). Um arquivo `viirs-snpp_2026_meta.json` guarda até que dia é SP e de que dia em diante é NRT.

In [3]:
def baixar(fonte, ini, fim):
    partes, d = [], ini
    while d <= fim:
        n = min(DAY_RANGE_MAX, (fim - d).days + 1)
        df = get_csv(f'area/csv/{KEY}/{fonte}/{BBOX}/{n}/{d.isoformat()}')
        if len(df):
            partes.append(df.assign(_fonte=fonte))
        d += timedelta(days=n)
        time.sleep(0.5)
    print(f'{fonte}: {ini} → {fim}: {sum(map(len, partes)):,} focos')
    return partes


def normaliza(df):
    df = df.copy()
    for c in COLS:
        if c not in df.columns:
            df[c] = pd.NA
    conf = df.confidence.astype(str).str.strip().str.lower()
    df['confidence'] = conf.map({'nominal': 'n', 'high': 'h', 'low': 'l'}).fillna(conf.str[:1])
    df['acq_time'] = df.acq_time.astype(str).str.strip().str.zfill(4)
    df['acq_date'] = pd.to_datetime(df.acq_date).dt.strftime('%Y-%m-%d')
    for c in ['latitude', 'longitude', 'frp']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['type'] = pd.to_numeric(df['type'], errors='coerce').astype('Int64')
    df['satellite'] = df.satellite.fillna('N').astype(str)
    df['daynight'] = df.daynight.astype(str)
    return df[COLS]


if KEY:
    (RAW / 'ano_corrente').mkdir(parents=True, exist_ok=True)
    partes = []
    sp_ate = min(SP_FIM, FIM) if (SP_FIM and SP_FIM >= INICIO) else None
    if sp_ate:
        partes += baixar('VIIRS_SNPP_SP', INICIO, sp_ate)
    nrt_ini = (sp_ate + timedelta(1)) if sp_ate else INICIO
    if nrt_ini <= FIM:
        partes += baixar('VIIRS_SNPP_NRT', nrt_ini, FIM)
    bruto = normaliza(pd.concat(partes, ignore_index=True)) if partes else pd.DataFrame(columns=COLS)
    bruto = bruto.drop_duplicates().reset_index(drop=True)
    bruto = bruto[pd.to_datetime(bruto.acq_date).dt.date.between(INICIO, FIM)]
    bruto.to_parquet(RAW / 'ano_corrente' / f'viirs-snpp_{ANO}.parquet', index=False)
    meta = {'sp_ate': sp_ate.isoformat() if sp_ate else None, 'nrt_desde': nrt_ini.isoformat() if nrt_ini <= FIM else None,
            'baixado_em': HOJE.isoformat(), 'bbox': BBOX}
    (RAW / 'ano_corrente' / f'viirs-snpp_{ANO}_meta.json').write_text(json.dumps(meta, indent=2))
    print(f'{len(bruto):,} focos salvos em data/raw/viirs-snpp_{ANO}.parquet'); print(meta)
    display(bruto.confidence.value_counts(dropna=False))
else:
    print('Sem MAP_KEY: pulando.')

VIIRS_SNPP_SP: 2026-01-01 → 2026-07-31: 295,192 focos


VIIRS_SNPP_NRT: 2026-08-01 → 2026-11-14: 694,107 focos


989,299 focos salvos em data/raw/viirs-snpp_2026.parquet
{'sp_ate': '2026-07-31', 'nrt_desde': '2026-08-01', 'baixado_em': '2026-11-14', 'bbox': '-75,-35,-28,6'}


confidence
n    846930
l     82700
h     59669
Name: count, dtype: int64

## 3 — Mesmo processamento do projeto + recorte do Brasil
- **Recorte do Brasil:** como a API é por retângulo, ele pega focos de países vizinhos. Mantemos só pontos cuja célula de **0,1°** (~11 km) já teve algum foco VIIRS nos arquivos **por país (Brasil)** de 2012–2025. Treze anos de VIIRS cobrem praticamente todo o território; o erro fica numa faixa de até ~11 km na fronteira. (Se os brutos 2012–2025 não estiverem na máquina, usa as células de 0,25° do `harmonized.parquet`, mais grosseiro.)
- **Filtros:** `confidence ∈ {n, h}`; remove pontos cuja célula de 0,02° (`fy = floor(lat/0.02)`, `fx = floor(lon/0.02)`) está em `static_mask.parquet`.
- **Grade:** `lat = floor(lat/0.25)*0.25`, idem `lon`, × ano × mês.
- **`dado_incompleto = True`** quando: é o **mês corrente** (provisório), **ou** algum dia do mês não tem nenhum registro no Brasil (mesma regra do notebook 04), (o mês ser servido por **NRT** não basta para marcar como incompleto: com todos os dias presentes ele é usado normalmente; o NRT é substituído pelo dado *standard* depois de ~2–3 meses, com diferenças pequenas — a coluna `usa_nrt` fica registrada nesta tabela).

In [4]:
if KEY:
    con = duckdb.connect()
    anteriores = sorted(p.as_posix() for p in RAW.glob('viirs-snpp_*.parquet')
                        if p.stem.split('_')[-1].isdigit() and int(p.stem.split('_')[-1]) < ANO)
    if anteriores:
        con.execute(f"""CREATE TABLE brasil AS SELECT DISTINCT floor(latitude/{CELULA_BR})::INT by_, floor(longitude/{CELULA_BR})::INT bx
                        FROM read_parquet({anteriores})""").fetchall()
        cond_br = f'(floor(latitude/{CELULA_BR})::INT, floor(longitude/{CELULA_BR})::INT) IN (SELECT (by_, bx) FROM brasil)'
        print(f'Pegada do Brasil: {con.execute("SELECT count(*) FROM brasil").fetchall()[0][0]:,} células de 0,1° ({len(anteriores)} anos de VIIRS)')
    else:
        con.execute(f"CREATE TABLE brasil AS SELECT DISTINCT lat, lon FROM read_parquet('{(PROC / 'harmonized.parquet').as_posix()}')").fetchall()
        cond_br = f'(floor(latitude/{CELULA})*{CELULA}, floor(longitude/{CELULA})*{CELULA}) IN (SELECT (lat, lon) FROM brasil)'
        print('Brutos 2012–2025 ausentes: usando células de 0,25° do harmonized.parquet como pegada do Brasil.')
    con.register('bruto', bruto)
    con.execute(f'CREATE TABLE br AS SELECT * FROM bruto WHERE {cond_br}').fetchall()
    n_tot, n_br = len(bruto), con.execute('SELECT count(*) FROM br').fetchall()[0][0]
    print(f'Focos no retângulo: {n_tot:,} · dentro do Brasil: {n_br:,}')

    gsat26 = con.execute(f"""
      SELECT floor(latitude/{CELULA})*{CELULA} lat, floor(longitude/{CELULA})*{CELULA} lon,
             year(CAST(acq_date AS DATE)) ano, month(CAST(acq_date AS DATE)) mes, 'S-NPP' satellite, daynight, count(*) n
      FROM br p
      ANTI JOIN read_parquet('{(PROC / 'static_mask.parquet').as_posix()}') m
        ON floor(p.latitude/{FINO})::INT = m.fy AND floor(p.longitude/{FINO})::INT = m.fx
      WHERE p.confidence IN ('n', 'h')
      GROUP BY ALL""").df()
    # Recorte final: só células 0,25° que já apareceram no Brasil em 2001–(ANO-1). As células novas de 2026 eram todas de
    # fronteira (Bolívia, Paraguai, Venezuela, Argentina...), sem nenhum foco em 25 anos de arquivos do Brasil (revisão de 14/11/2026).
    conhecidas = pd.read_parquet(PROC / 'harmonized.parquet', columns=['lat', 'lon', 'ano']).query('ano < @ANO')[['lat', 'lon']].drop_duplicates()
    antes_corte = gsat26.n.sum()
    gsat26 = gsat26.merge(conhecidas, on=['lat', 'lon'])
    print(f'Recorte às células conhecidas do Brasil: removidos {antes_corte - gsat26.n.sum():,} focos de células novas de fronteira')
    gsat26['bloco_lat'], gsat26['bloco_lon'] = (gsat26.lat // 5) * 5, (gsat26.lon // 5) * 5
    gsat26.to_parquet(PROC / f'grid_sat_{ANO}.parquet', index=False)
    grid26 = gsat26.groupby(['lat', 'lon', 'ano', 'mes'], as_index=False).n.sum()

    # meses incompletos
    dias = con.execute('SELECT month(CAST(acq_date AS DATE)) mes, count(DISTINCT acq_date) dias FROM br GROUP BY 1').df()
    dias = dias.set_index('mes').dias.reindex(range(1, FIM.month + 1), fill_value=0).rename_axis('mes').reset_index()
    dias['dias_no_mes'] = [calendar.monthrange(ANO, m)[1] for m in dias.mes]
    dias['dias_esperados'] = [(FIM - date(ANO, m, 1)).days + 1 if m == FIM.month else d for m, d in zip(dias.mes, dias.dias_no_mes)]
    nrt_desde = date.fromisoformat(meta['nrt_desde']) if meta['nrt_desde'] else None
    dias['usa_nrt'] = [bool(nrt_desde) and nrt_desde <= date(ANO, m, calendar.monthrange(ANO, m)[1]) for m in dias.mes]
    dias['mes_corrente'] = [(ANO, m) == (HOJE.year, HOJE.month) for m in dias.mes]
    dias['dado_incompleto'] = dias.mes_corrente | (dias.dias < dias.dias_no_mes)   # NRT com todos os dias conta como completo (ver texto)
    display(dias)
    inc = dict(zip(dias.mes, dias.dado_incompleto))

    h26 = grid26[grid26.n > 0].rename(columns={'n': 'focos_harmonizados'})
    h26['focos_harmonizados'] = h26.focos_harmonizados.astype(float)
    h26['ano'] = h26.ano.astype('int64'); h26['mes'] = h26.mes.astype('int64')
    h26['fonte'] = 'viirs'
    h26['dado_incompleto'] = h26.mes.map(inc).fillna(True).astype(bool)
    h26 = h26[['lat', 'lon', 'ano', 'mes', 'focos_harmonizados', 'fonte', 'dado_incompleto']].sort_values(['ano', 'mes', 'lat', 'lon']).reset_index(drop=True)

    # checagens do contrato
    ref = pd.read_parquet(PROC / 'harmonized.parquet')
    assert list(h26.columns) == list(ref.columns)
    assert not h26.duplicated(['lat', 'lon', 'ano', 'mes']).any() and h26.focos_harmonizados.gt(0).all()
    assert (h26.ano == ANO).all() and h26.fonte.eq('viirs').all()
    h26.to_parquet(PROC / f'harmonized_{ANO}.parquet', index=False)
    print(f'harmonized_{ANO}.parquet: {len(h26):,} linhas')
    resumo = h26.groupby('mes').agg(focos=('focos_harmonizados', 'sum'), celulas=('lat', 'size'), incompleto=('dado_incompleto', 'first'))
    resumo['focos_2025_mesmo_mes'] = ref[ref.ano == ANO - 1].groupby('mes').focos_harmonizados.sum()
    display(resumo)
else:
    print('Sem MAP_KEY: pulando.')

Pegada do Brasil: 59,212 células de 0,1° (14 anos de VIIRS)


Focos no retângulo: 989,299 · dentro do Brasil: 679,519


Recorte às células conhecidas do Brasil: removidos 425 focos de células novas de fronteira


,mes,dias,dias_no_mes,dias_esperados,usa_nrt,mes_corrente,dado_incompleto
0,1,31,31,31,False,False,False
1,2,28,28,28,False,False,False
2,3,28,31,31,False,False,True
3,4,27,30,30,False,False,True
4,5,31,31,31,False,False,False
5,6,28,30,30,False,False,True
6,7,25,31,31,False,False,True
7,8,30,31,31,True,False,True
8,9,30,30,30,True,False,False
9,10,31,31,31,True,False,False


harmonized_2026.parquet: 39,446 linhas


,focos,celulas,incompleto,focos_2025_mesmo_mes
mes,,,,
1,21643.0,3556,False,14656.0
2,11184.0,2393,False,9404.0
3,10220.0,2046,True,11796.0
4,7045.0,1920,True,7529.0
5,18516.0,2754,False,23843.0
6,25751.0,3149,True,32928.0
7,30928.0,4125,True,48630.0
8,125575.0,5853,True,119564.0
9,160927.0,6295,False,188427.0


## 4 — Anexar 2026 ao `harmonized.parquet`
`anexar_2026()` apaga as linhas de 2026 que já existirem no `harmonized.parquet` e junta as de `harmonized_2026.parquet` (pode rodar várias vezes sem duplicar). Só é chamada se `ANEXAR = True` no topo.

In [5]:
def anexar_2026():
    alvo, novo = PROC / 'harmonized.parquet', PROC / f'harmonized_{ANO}.parquet'
    h, h26 = pd.read_parquet(alvo), pd.read_parquet(novo)
    assert list(h26.columns) == list(h.columns), 'colunas fora do contrato'
    antes = int((h.ano == ANO).sum())
    h = pd.concat([h[h.ano != ANO], h26], ignore_index=True).sort_values(['ano', 'mes', 'lat', 'lon']).reset_index(drop=True)
    assert not h.duplicated(['lat', 'lon', 'ano', 'mes']).any()
    tmp = alvo.with_suffix('.tmp.parquet')
    h.to_parquet(tmp, index=False); tmp.replace(alvo)
    print(f'harmonized.parquet: removidas {antes:,} linhas antigas de {ANO}, anexadas {len(h26):,}; total {len(h):,} ({h.ano.min()}–{h.ano.max()})')
    # mesma coisa para a grade por satélite (usada pelo mapa de observações do app)
    gs, g26 = pd.read_parquet(PROC / 'grid_sat.parquet'), pd.read_parquet(PROC / f'grid_sat_{ANO}.parquet')
    g26 = g26[gs.columns].astype(gs.dtypes.to_dict())
    gs = pd.concat([gs[gs.ano != ANO], g26], ignore_index=True)
    tmp = (PROC / 'grid_sat.parquet').with_suffix('.tmp.parquet'); gs.to_parquet(tmp, index=False); tmp.replace(PROC / 'grid_sat.parquet')
    print(f'grid_sat.parquet: anexadas {len(g26):,} linhas de {ANO}; anos {gs.ano.min()}–{gs.ano.max()}')


if ANEXAR and KEY and (PROC / f'harmonized_{ANO}.parquet').exists():
    anexar_2026()
else:
    print('ANEXAR = False (ou sem MAP_KEY): harmonized.parquet não foi alterado.')

harmonized.parquet: removidas 39,510 linhas antigas de 2026, anexadas 39,446; total 1,252,185 (2001–2026)


grid_sat.parquet: anexadas 52,406 linhas de 2026; anos 2001–2026
